In [30]:
# !pip install transformers torch # 최초 1회 설치 필요

import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np

In [31]:
# 1. 사전학습된 BERT 불러오기
# bert-base-uncased: 영어 소문자기준 BERT 기본 모델
# 위키피디아 + 책 데이터로 미리 학습된 상태 -> 바로 쓰면 됨
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name) # 텍스트 -> 토큰 변환기
model = AutoModel.from_pretrained(model_name)          # 토큰 -> 벡터 변환 모델
model.eval()    # 추론 모드 (학습 X)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [32]:
# 2. BERT의 토큰화 - TF-IDF 떄와 다른 점
text = "this movie was unbelievably good"
tokens = tokenizer.tokenize(text)
print('토큰:', tokens)
# 주목:: 'unbelievably' 같은 긴 단어가 조각(subword)으로 쪼개짐
# ex) ['un', '##bel', '##ie', '##va', '##bli'] 형태
# -> 처음 보는 단어도 조각으로 나눠서 처리 가능 (TF-IDF는 모르는 단어면 그냥 무시)

# 실제 모델 입력 형태로 변환
encoded = tokenizer(text, return_tensors='pt')
print('입력 ID:', encoded['input_ids'])
# 101(=[CLS] 문장 시작), 102(=[SEP] 문장 끝) 특수 토큰이 자동으로 붙음

토큰: ['this', 'movie', 'was', 'un', '##bel', '##ie', '##va', '##bly', 'good']
입력 ID: tensor([[ 101, 2023, 3185, 2001, 4895, 8671, 2666, 3567, 6321, 2204,  102]])


In [33]:
# 3. 핵심 실험: 같은 단어, 다른 문맥 -> 다른 벡터?
def get_word_vector(sentence, target_word):
  """문장 안에서 특정 단어의 BERT 벡터(문백 반영)를 추출"""
  encoded = tokenizer(sentence, return_tensors='pt')
  with torch.no_grad():       # 학습 안 하므로 기울기 계산 끔 (속도↑)
    output = model(**encoded)
  # last_hidden_state: 각 토큰의 최종 벡터 (1, 토큰수, 768차원)
  hidden = output.last_hidden_state[0]

  # target_word가 몇 번째 토큰인지 찾기
  tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])

  target_tokens = tokenizer.tokenize(target_word)

  # 전체 토큰 스퀀스에서 target_tokens와 일치하는 구간 찾기
  for i in range(len(tokens) - len(target_tokens) + 1):
    if tokens[i:i+len(target_tokens)] == target_tokens:
      # 해당 구간 벡터들을 평균 내서 하나의 벡터로
      return hidden[i:i+len((target_tokens))].mean(dim=0).numpy()
  raise ValueError(f"'{target_word}' 관련 토큰을 찾지 못함")

def cosine_sim(a, b):
  """두 벡터의 코사인 유사도 (1에 가까울수록 비슷)"""
  return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# 같은 단어 'good'이 서로 다른 문맥에 놓였을 때
s1 = "this movie was so good"      # 긍정 맥락
s2 = "this movie was not good"     # 부정 맥락
s3 = "the food tasted really good" # 다른 주제, 긍정 맥락

v1 = get_word_vector(s1, 'good')
v2 = get_word_vector(s2, 'good')
v3 = get_word_vector(s3, 'good')

print('so good vs not good:', cosine_sim(v1, v2))
print('so good vs tasted good:', cosine_sim(v1, v3))
# 정적 임베딩(Glove)이었다면 셋 다 완전히 동일한 벡터라 유사도가 전부 1.0
# BERT는 문맥에 따라 다른 값이 나옴 -> 문맥을 반영한다는 증거

so good vs not good: 0.71976846
so good vs tasted good: 0.73514885


In [34]:
# 4. 문장 전체를 하나의 벡터로 (문장 임베딩)
def get_sentence_vector(sentence):
  """문장 전체를 대표하는 벡터 (모든 토큰 벡터의 평균)"""
  encoded = tokenizer(sentence, return_tensors='pt')
  with torch.no_grad():
    output = model(**encoded)
  return output.last_hidden_state[0].mean(dim=0).numpy()  # 토큰 평균

a = get_sentence_vector("this film was fantastic")
b = get_sentence_vector("the movie was excellent")   # 의미 비슷
c = get_sentence_vector("this film was terrible")    # 의미 반대

print('fantastic vs excellent:', cosine_sim(a, b))   # 높게 나와야 정상
print('fantastic vs terrible:', cosine_sim(a, c))   # 낮게 나와야 정상

fantastic vs excellent: 0.8500998
fantastic vs terrible: 0.93170124


In [35]:
# 5. TODO
# 5-1. 'bank' 벡터 유사도를 비교
t1 = get_word_vector("i went to the bank to deposit money", 'bank')  # (은행)
t2 = get_word_vector("we sat on the river bank", 'bank')             # (강둑)
print(cosine_sim(t1, t2))
print("-"*40)

# 5-2. happy/sad -> 문장 임베딩으로 비교
t3 = get_sentence_vector("i feel very happy today")
t4 = get_sentence_vector("i feel very joyful today")
t5 = get_sentence_vector("i feel very sad today")
print('happy vs joyful:', cosine_sim(t3, t4))
print('happy vs sad:', cosine_sim(t3, t5))
print("-"*40)

# 추가 (단어 벡터만 직접 비교)
v_happy = get_word_vector("i feel very happy today", 'happy')
v_joyful = get_word_vector("i feel very joyful today", 'joy')
v_sad = get_word_vector("i feel very sad today", 'sad')

print('* word_vector')
print('happy vs joyful:', cosine_sim(v_happy, v_joyful))
print('happy vs sad:', cosine_sim(v_happy, v_sad))
print("-"*40)

# 5-3. 네 개의 단어를 tokenizer.tokenize()로 어떻게 쪼개지는지 확인
# `text`가 문자열이므로, 리스트로 변경하여 단어를 추가합니다.
text_list = ["playing", "unhappiness", "tokenization", "pretrained"]
for word in text_list:
  print(f"'{word}': {tokenizer.tokenize(word)}")

0.3402438
----------------------------------------
happy vs joyful: 0.91507536
happy vs sad: 0.9204229
----------------------------------------
* word_vector
happy vs joyful: 0.75685304
happy vs sad: 0.6913599
----------------------------------------
'playing': ['playing']
'unhappiness': ['un', '##ha', '##pp', '##iness']
'tokenization': ['token', '##ization']
'pretrained': ['pre', '##train', '##ed']
